In [ ]:
%cd /content
!rm -rf AML_HW1
!git clone https://github.com/kmb3322/26sAML_HW1.git AML_HW1
%cd /content/AML_HW1
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cse493s_hw1_runs
!ln -sfn /content/drive/MyDrive/cse493s_hw1_runs /content/AML_HW1/runs

!ls runs

In [ ]:
# Sanity check
%cd /content/AML_HW1
!python train.py --config configs/sanity.yaml
!python train.py --config configs/sanity_masked.yaml
!python inference.py --checkpoint_dir runs/sanity_masked/final --sanity_prompt "I love"

In [ ]:
%cd /content/AML_HW1

!python train.py --config configs/add_p97_l1_seed0.yaml
!python train.py --config configs/add_p97_l2_seed0.yaml
!python train.py --config configs/add_p113_l1_seed0.yaml
!python train.py --config configs/add_p113_l2_seed0.yaml

!python train.py --config configs/sub_p97_l1_seed0.yaml
!python train.py --config configs/sub_p97_l2_seed0.yaml
!python train.py --config configs/sub_p113_l1_seed0.yaml
!python train.py --config configs/sub_p113_l2_seed0.yaml

!python train.py --config configs/div_p97_l1_seed0.yaml
!python train.py --config configs/div_p97_l2_seed0.yaml
!python train.py --config configs/div_p113_l1_seed0.yaml
!python train.py --config configs/div_p113_l2_seed0.yaml

# random restarts: same config, different seed & out_dir
!python train.py --config configs/add_p97_l1_seed0.yaml --seed 1 --out_dir runs/add_p97_l1_seed1
!python train.py --config configs/add_p97_l1_seed0.yaml --seed 2 --out_dir runs/add_p97_l1_seed2

# addition inference
!python inference.py --checkpoint_dir runs/add_p97_l1_seed0/best --a 12 --b 34 --op + --p 97

**1.3.Grokking**

In [ ]:
%cd /content/AML_HW1
!python train.py --config configs/div_p97_grokking_seed0.yaml

!python inference.py --checkpoint_dir runs/div_p97_grokking_seed0/best --a 3 --b 5 --op / --p 97

**1.4.ablations**

In [ ]:
%cd /content/AML_HW1

!python train.py --config configs/abl_wd_0.0.yaml
!python train.py --config configs/abl_wd_0.1.yaml
!python train.py --config configs/abl_wd_0.5.yaml

!python train.py --config configs/abl_tf_0.30.yaml
!python train.py --config configs/abl_tf_0.40.yaml
!python train.py --config configs/abl_tf_0.60.yaml
!python train.py --config configs/abl_tf_0.80.yaml

**Cross-run plots**

In [ ]:
!mkdir -p /content/AML_HW1/runs/plots

In [ ]:
#Random-restart overlay (Part 1.2)

import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

PLOT_ROOT = Path("runs/plots")
(PLOT_ROOT / "restarts").mkdir(parents=True, exist_ok=True)
plt.figure()
for s in (0, 1, 2):
    df = pd.read_csv(f"runs/add_p97_l1_seed{s}/metrics.csv")
    plt.plot(df["step"], df["train_acc"], alpha=0.7, label=f"seed{s} train")
    plt.plot(df["step"], df["test_acc"], alpha=0.7, ls="--", label=f"seed{s} test")
plt.xlabel("step"); plt.ylabel("accuracy"); plt.ylim(-0.02, 1.02)
plt.title("Random restarts: + p=97 1-layer"); plt.legend(fontsize=8)
plt.tight_layout(); plt.savefig(PLOT_ROOT / "restarts" / "seeds_acc.png", dpi=200)

In [ ]:
# Grokking annotated (Part 1.3)

import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

PLOT_ROOT = Path("runs/plots")
THR = 0.99
df = pd.read_csv("runs/div_p97_grokking_seed0/metrics.csv")
def first(col):
    c = df[df[col] >= THR]
    return None if c.empty else int(c["step"].iloc[0])
ts, vs = first("train_acc"), first("test_acc")

(PLOT_ROOT / "grokking").mkdir(parents=True, exist_ok=True)
plt.figure()
plt.plot(df["step"], df["train_acc"], label="train")
plt.plot(df["step"], df["test_acc"],  label="test")
if ts: plt.axvline(ts, ls=":", color="C0", alpha=0.6, label=f"train≥{THR} @ {ts}")
if vs: plt.axvline(vs, ls=":", color="C1", alpha=0.6, label=f"test≥{THR}  @ {vs}")
plt.xscale("log"); plt.ylim(-0.02, 1.02)
plt.xlabel("step"); plt.ylabel("accuracy")
plt.title("Grokking: / p=97 (AdamW, wd=1.0)"); plt.legend(fontsize=9)
plt.tight_layout(); plt.savefig(PLOT_ROOT / "grokking" / "grokking_acc.png", dpi=200)
print(f"train_solved={ts}  test_solved={vs}  delay={None if vs is None or ts is None else vs-ts}")

In [ ]:
# Ablation comparison + summary table (Part 1.4)

import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

PLOT_ROOT = Path("runs/plots")
ABLATIONS = {
    "weight_decay": [
        ("abl_wd_0.0", "wd=0.0"),
        ("abl_wd_0.1", "wd=0.1"),
        ("abl_wd_0.5", "wd=0.5"),
        ("div_p97_grokking_seed0", "wd=1.0 (baseline)"),
    ],
    "train_frac": [
        ("abl_tf_0.30", "tf=0.30"),
        ("abl_tf_0.40", "tf=0.40"),
        ("div_p97_grokking_seed0", "tf=0.50 (baseline)"),
        ("abl_tf_0.60", "tf=0.60"),
        ("abl_tf_0.80", "tf=0.80"),
    ],
}
THR = 0.99
def first(df, col):
    c = df[df[col] >= THR]
    return None if c.empty else int(c["step"].iloc[0])

rows = []
for name, runs in ABLATIONS.items():
    (PLOT_ROOT / "ablation" / name).mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(8, 5))
    for run, label in runs:
        csv = Path(f"runs/{run}/metrics.csv")
        if not csv.exists():
            print("skip", csv); continue
        df = pd.read_csv(csv)
        plt.plot(df["step"], df["train_acc"], alpha=0.4, ls="--", label=f"{label} train")
        plt.plot(df["step"], df["test_acc"],  alpha=0.9,         label=f"{label} test")
        ts, vs = first(df, "train_acc"), first(df, "test_acc")
        rows.append({
            "ablation": name, "run": run, "label": label,
            "train_solved_step": ts, "test_solved_step": vs,
            "delay": (vs - ts) if (ts is not None and vs is not None) else None,
        })
    plt.xscale("log"); plt.ylim(-0.02, 1.02)
    plt.xlabel("step"); plt.ylabel("accuracy")
    plt.title(f"Ablation: {name} (division p=97, 2-layer)")
    plt.legend(fontsize=8, ncol=2); plt.tight_layout()
    plt.savefig(PLOT_ROOT / "ablation" / name / "comparison_acc.png", dpi=200)
    plt.close()

pd.DataFrame(rows).to_csv(PLOT_ROOT / "ablation" / "summary.csv", index=False)
print(pd.DataFrame(rows))